# 1. Registers and memory

We will use a small simulated processor to see how numbers are stored and how instructions change them. Start with bits and bytes, then step through a program that adds two numbers.

Run the cells in order. Before each example, try to work out the result yourself.


## 0. Setup

Create your Python environment using the [README](README.md), then select it as the notebook kernel. Run the cell below with **Shift+Enter**. Run it again after restarting the kernel.

Python controls the simulator. The text inside `CPU("""...""")` is a MINI-8 assembly program. `show()` displays the processor, `step()` runs one instruction, and `run()` continues until `HALT`.


In [ ]:
from pathlib import Path
import sys

# Locate the simulator module.
candidates = (Path.cwd(), Path.cwd().parent)
project_dir = next(
    (path for path in candidates
     if (path / "mini8.py").is_file() and (path / "notebook_tools.py").is_file()),
    None,
)
if project_dir is None:
    raise RuntimeError("Open the project containing mini8.py and restart the kernel.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from notebook_tools import CPU
from mini8 import assemble, disassemble

print("MINI-8 is ready.")


MINI-8 is ready.


### 0.1. Bits and bytes

A **bit** is either `0` or `1`. A **byte** has eight bits. Their place values, from left to right, are 128, 64, 32, 16, 8, 4, 2, and 1. Add the values wherever a bit is `1`: `00000101` means 4 + 1 = 5.

MINI-8 treats bytes as unsigned numbers: they range from 0 to 255. What number is `00001010`?

The Python code below prints numbers in binary. In `f"{value:08b}"`, `08b` means “use eight binary digits, adding zeros on the left.”


In [21]:
print("Bit position:  7   6   5   4   3   2   1   0")
print("Place value: 128  64  32  16   8   4   2   1")
print()
for value in (3, 4, 5, 8, 10, 255):
    print(f"{value:3d} = {value:08b}")

print("00000101 =", 4 + 1)
print("00001010 =", 8 + 2)


Bit position:  7   6   5   4   3   2   1   0
Place value: 128  64  32  16   8   4   2   1

  3 = 00000011
  4 = 00000100
  5 = 00000101
  8 = 00001000
 10 = 00001010
255 = 11111111
00000101 = 5
00001010 = 10


### 0.2. Binary addition

Add from right to left, just as in decimal. If a column adds up to 2 (binary `10`), write `0` and carry `1`. If it adds up to 3 (binary `11`), write `1` and carry `1`.

Try 5 + 3 in binary: `00000101 + 00000011`. The last table below shows each column, starting at the rightmost bit.

A sum can need more than eight bits. For 250 + 10 = 260, compare the full sum with the stored byte. This Python example shows the carries; MINI-8 itself has no carry flag.


In [22]:
for left, right in ((3, 4), (5, 3), (250, 10)):
    full_result = left + right
    stored_result = full_result % 256
    print(f"{left:08b} + {right:08b}")
    print(f"Full sum:    {full_result:09b} = {full_result}")
    print(f"Stored byte:  {stored_result:08b} = {stored_result}")
    print()

left, right = 5, 3
carry = 0
print("Carry trace for 5 + 3, starting at the rightmost bit:")
print("Position | Left | Right | Carry in | Result bit | Carry out")
for position in range(8):
    left_bit = (left >> position) & 1
    right_bit = (right >> position) & 1
    column_total = left_bit + right_bit + carry
    result_bit = column_total % 2
    next_carry = column_total // 2
    print(f"{position:8d} | {left_bit:4d} | {right_bit:5d} | {carry:8d} | {result_bit:10d} | {next_carry:9d}")
    carry = next_carry


00000011 + 00000100
Full sum:    000000111 = 7
Stored byte:  00000111 = 7

00000101 + 00000011
Full sum:    000001000 = 8
Stored byte:  00001000 = 8

11111010 + 00001010
Full sum:    100000100 = 260
Stored byte:  00000100 = 4

Carry trace for 5 + 3, starting at the rightmost bit:
Position | Left | Right | Carry in | Result bit | Carry out
       0 |    1 |     1 |        0 |          0 |         1
       1 |    0 |     1 |        1 |          0 |         1
       2 |    1 |     0 |        1 |          0 |         1
       3 |    0 |     0 |        1 |          1 |         0
       4 |    0 |     0 |        0 |          0 |         0
       5 |    0 |     0 |        0 |          0 |         0
       6 |    0 |     0 |        0 |          0 |         0
       7 |    0 |     0 |        0 |          0 |         0


### 0.3. What happens after 255?

An eight-bit register holds values from 0 to 255. MINI-8 keeps only the lowest eight bits of a result:

- 250 + 10 = 260, but the register stores 4 (`00000100`).
- 0 − 1 wraps around to 255 (`11111111`).

This is arithmetic modulo 256. Here, 255 is a positive value.

Run the example and check the register values and output. We will go through the instructions in sections 1–4. The `CMP` display is for comparisons; it does not show a carry.


In [23]:
overflow_cpu = CPU("""
LDI R0, 250
LDI R1, 10
ADD R0, R1
OUT R0
LDI R0, 0
LDI R1, 1
SUB R0, R1
OUT R0
HALT
""", title="250 + 10 and 0 − 1 in eight bits")
overflow_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 250",0 → 1,250,0,0,0,=,—,—
2,"LDI R1, 10",1 → 2,250,10,0,0,=,—,—
3,"ADD R0, R1",2 → 3,4,10,0,0,=,—,—
4,OUT R0,3 → 4,4,10,0,0,=,—,4
5,"LDI R0, 0",4 → 5,0,10,0,0,=,—,—
6,"LDI R1, 1",5 → 6,0,1,0,0,=,—,—
7,"SUB R0, R1",6 → 7,255,1,0,0,=,—,—
8,OUT R0,7 → 8,255,1,0,0,=,—,255
9,HALT,8 → 8 · HALT,255,1,0,0,=,—,—


### 0.4. Compare with Python

Python integers can grow beyond eight bits. `% 256` gives the remainder after division by 256, so we can use it to check the MINI-8 results.

The first line below prints 260 and −1. The second prints 4 and 255.


In [24]:
print("Python:", 250 + 10, 0 - 1)
print("Modulo 256:", (250 + 10) % 256, (0 - 1) % 256)


Python: 260 -1
Modulo 256: 4 255


## 1. Load a program

`R0` through `R3` are the processor's four **registers**. Each stores one byte. `PC` holds the number of the next instruction, starting at 0. We will use the `CMP` flag in the second notebook.

`CPU(...)` loads a program and starts with all registers at zero. `show()` displays the state without running anything. Does `R0` already contain 3?


In [25]:
cpu = CPU("""
LDI R0, 3
LDI R1, 4
ADD R0, R1
OUT R0
HALT
""", title="First program: 3 + 4")
cpu.show()


## 2. LDI: load a number

`LDI R0, 3` puts 3 into `R0` and leaves the other registers unchanged.

Run one step below. The table shows the registers after the instruction. In the `PC before → after` column, the left number is the instruction just run; the right number is the next one.


In [26]:
cpu.step()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 3",0 → 1,3,0,0,0,=,—,—


## 3. ADD: add two registers

The next two instructions load 4 into `R1`, then add it to `R0`.

`ADD R0, R1` stores the sum in `R0`. `R1` keeps its value. What should the two registers contain after these steps?


In [27]:
cpu.step(count=2)


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
1,Initial state,1,3,0,0,0,=,—,—
2,"LDI R1, 4",1 → 2,3,4,0,0,=,—,—
3,"ADD R0, R1",2 → 3,7,4,0,0,=,—,—


## 4. OUT and HALT: print and stop

`OUT R0` prints the value in `R0` without changing the register. `HALT` stops the program.

`run()` continues from the current instruction. After stopping, `PC` stays at `HALT`.


In [28]:
cpu.run()
print("Output:", cpu.output)


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
3,Initial state,3,7,4,0,0,=,—,—
4,OUT R0,3 → 4,7,4,0,0,=,—,7
5,HALT,4 → 4 · HALT,7,4,0,0,=,—,—


Output: (7,)


### 4.1. Run the program again

`reset()` keeps the program but clears its state: registers, data memory, step count, and output. We can then run it from the beginning.

Running a `step()` cell twice runs two instructions. To start over, use `reset()` or rerun the cell containing `CPU(...)`. To restart the whole notebook, use **Restart Kernel → Run All**.


In [29]:
cpu.reset()
cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 3",0 → 1,3,0,0,0,=,—,—
2,"LDI R1, 4",1 → 2,3,4,0,0,=,—,—
3,"ADD R0, R1",2 → 3,7,4,0,0,=,—,—
4,OUT R0,3 → 4,7,4,0,0,=,—,7
5,HALT,4 → 4 · HALT,7,4,0,0,=,—,—


## 5. MOV and SUB: copy and subtract

`MOV R1, R0` copies `R0` into `R1`. `SUB R0, R2` subtracts `R2` from `R0` and stores the result in `R0`.

This example creates a new processor. After the subtraction, does the copy in `R1` change too?


In [30]:
copy_cpu = CPU("""
LDI R0, 9
MOV R1, R0
LDI R2, 2
SUB R0, R2
OUT R0
OUT R1
HALT
""", title="Copying a value does not link the registers")
copy_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 9",0 → 1,9,0,0,0,=,—,—
2,"MOV R1, R0",1 → 2,9,9,0,0,=,—,—
3,"LDI R2, 2",2 → 3,9,9,2,0,=,—,—
4,"SUB R0, R2",3 → 4,7,9,2,0,=,—,—
5,OUT R0,4 → 5,7,9,2,0,=,—,7
6,OUT R1,5 → 6,7,9,2,0,=,—,9
7,HALT,6 → 6 · HALT,7,9,2,0,=,—,—


## 6. STORE and LOAD: use data memory

MINI-8 has 256 data cells, with addresses from 0 to 255. `[10]` means the contents of the cell at address 10.

- `STORE [10], R0` copies `R0` into cell 10.
- `LOAD R1, [10]` copies cell 10 into `R1`.

Data memory is separate from the program: storing a value in cell 10 leaves instruction 10 unchanged.

Run the first two instructions to put 42 into cell 10.


In [31]:
memory_cpu = CPU("""
LDI R0, 42
STORE [10], R0
LDI R0, 0
LOAD R1, [10]
OUT R1
HALT
""", watch=(10,), title="Register → data memory → register")
memory_cpu.step(count=2)


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 42",0 → 1,42,0,0,0,=,—,—
2,"STORE [10], R0",1 → 2,42,0,0,0,=,[10]: 0 → 42,—


### 6.1. Read the stored value

The next two instructions put 0 into `R0`, then load cell 10 into `R1`. What will `R1` contain?

Compare `R0`, `R1`, and cell 10. Both `STORE` and `LOAD` copy a value and leave the source unchanged.


In [32]:
memory_cpu.step(count=2)


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
2,Initial state,2,42,0,0,0,=,—,—
3,"LDI R0, 0",2 → 3,0,0,0,0,=,—,—
4,"LOAD R1, [10]",3 → 4,0,42,0,0,=,—,—


### 6.2. Look at nearby cells

Finish the program and display cells 8 through 15. `stop=16` excludes cell 16. Did any of the cells next to cell 10 change?


In [33]:
memory_cpu.run()
memory_cpu.memory(start=8, stop=16)


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
4,Initial state,4,0,42,0,0,=,—,—
5,OUT R1,4 → 5,0,42,0,0,=,—,42
6,HALT,5 → 5 · HALT,0,42,0,0,=,—,—


## 7. A value or an address?

`LDI R1, 10` loads the number 10. `LOAD R2, [10]` loads whatever is stored at address 10.

Run the example, then change 42 to 17 in the first instruction and rerun it. Which output changes?


In [34]:
address_cpu = CPU("""
LDI R0, 42
STORE [10], R0
LDI R1, 10
LOAD R2, [10]
OUT R1
OUT R2
HALT
""", watch=(10,), title="Value versus address")
address_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 42",0 → 1,42,0,0,0,=,—,—
2,"STORE [10], R0",1 → 2,42,0,0,0,=,[10]: 0 → 42,—
3,"LDI R1, 10",2 → 3,42,10,0,0,=,—,—
4,"LOAD R2, [10]",3 → 4,42,10,42,0,=,—,—
5,OUT R1,4 → 5,42,10,42,0,=,—,10
6,OUT R2,5 → 6,42,10,42,0,=,—,42
7,HALT,6 → 6 · HALT,42,10,42,0,=,—,—


## 8. Name an address

`.equ x, 10` makes `x` another name for the number 10. We use it here as a memory address. This line is an assembler directive: it takes no instruction slot and stores nothing in memory.

A new processor starts with zero in every data cell. The program reads `[x]` before storing 3 there. What are the two output values?


In [35]:
variable_cpu = CPU("""
.equ x, 10
LOAD R0, [x]
OUT R0
LDI R0, 3
STORE [x], R0
LOAD R1, [x]
OUT R1
HALT
""", watch=(10,), title="A variable's name and value")
variable_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LOAD R0, [10]",0 → 1,0,0,0,0,=,—,—
2,OUT R0,1 → 2,0,0,0,0,=,—,0
3,"LDI R0, 3",2 → 3,3,0,0,0,=,—,—
4,"STORE [10], R0",3 → 4,3,0,0,0,=,[10]: 0 → 3,—
5,"LOAD R1, [10]",4 → 5,3,3,0,0,=,—,—
6,OUT R1,5 → 6,3,3,0,0,=,—,3
7,HALT,6 → 6 · HALT,3,3,0,0,=,—,—


## 9. Try it: calculate, store, and load

Change the program below to:

1. Calculate (8 + 5) − 3.
2. Store the result in cell 20.
3. Load it into `R2` and print `R2`.

The starting program only adds 8 + 5 and prints 13. Your version should print 10 and leave 10 in cell 20. `watch=(20,)` shows that cell in the display.


In [36]:
exercise_cpu = CPU("""
LDI R0, 8
LDI R1, 5
ADD R0, R1
OUT R0
HALT
""", watch=(20,), title="My solution")
exercise_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 8",0 → 1,8,0,0,0,=,—,—
2,"LDI R1, 5",1 → 2,8,5,0,0,=,—,—
3,"ADD R0, R1",2 → 3,13,5,0,0,=,—,—
4,OUT R0,3 → 4,13,5,0,0,=,—,13
5,HALT,4 → 4 · HALT,13,5,0,0,=,—,—


### 9.1. Check your solution

Which instruction writes to memory? Why is `.equ` alone not enough to store a value?

<details>
<summary>One possible solution</summary>

```asm
LDI R0, 8
LDI R1, 5
ADD R0, R1
LDI R1, 3
SUB R0, R1
STORE [20], R0
LOAD R2, [20]
OUT R2
HALT
```

`R0`, `R2`, and cell 20 all contain 10. `STORE` writes the value to memory; `.equ` only gives a number a name.

</details>

Section 10 shows how instructions become bytes. The [second notebook](02_conditionals_and_loops.ipynb) covers conditions and loops.


## 10. Instructions as bytes

The assembler turns each instruction into three bytes: `[opcode, A, B]`. The opcode identifies the instruction; `A` and `B` hold its operands. An unused operand byte is zero. For example, `LDI R0, 3` becomes `01 00 03` in hexadecimal.

Hexadecimal uses digits 0–9 and A–F, where A–F mean 10–15. Two hex digits represent one byte: `0D` is decimal 13.

Compare the instructions with their bytes below. `PC` counts instructions, so instruction 2 starts at byte 6. This five-instruction program takes 15 bytes.


In [37]:
source = """
LDI R0, 3
LDI R1, 4
ADD R0, R1
OUT R0
HALT
"""
binary_cpu = CPU(source, title="Instructions and their bytes")
binary_cpu.listing()
print("Bytes:", assemble(source).hex(" "))
print("Program size:", len(assemble(source)), "bytes")


Instruction number (PC),Byte offset,Opcode,A,B,Decoded instruction
0 ← PC,0,01,00,03,"LDI R0, 3"
1,3,01,01,04,"LDI R1, 4"
2,6,05,00,01,"ADD R0, R1"
3,9,0D,00,00,OUT R0
4,12,00,00,00,HALT


Bytes: 01 00 03 01 01 04 05 00 01 0d 00 00 00 00 00
Program size: 15 bytes


### 10.1. Convert bytes back to assembly

`disassemble(...)` converts bytes back to assembly with numeric operands. The original names and comments are lost when assembling. Disassembly does not run the program.

Find the `ADD` instruction below. Then change a number in the previous cell and compare the bytes and assembly again.


In [38]:
print(disassemble(assemble(source)))


LDI R0, 3              ; PC=000  01 00 03
LDI R1, 4              ; PC=001  01 01 04
ADD R0, R1             ; PC=002  05 00 01
OUT R0                 ; PC=003  0D 00 00
HALT                   ; PC=004  00 00 00

